# Preparing Data for Streamlit

In [1]:
# Imports

# General

import numpy as np
import pandas as pd
import geopandas as gpd
import math
import json
import re
import string

# Plotting

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook_connected' # For plotly graphs to render in this environment


# Scikit-Learn

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

In [2]:
# Set directory

PATH = "C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA"

In [3]:
# Load clustered builds

builds_df = pd.read_csv(f'{PATH}/CLUSTERED/builds_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(builds_df)

In [4]:
# Load clustered demos

demos_df = pd.read_csv(f'{PATH}/CLUSTERED/demos_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(demos_df)

In [5]:
# Load clustered renos

renos_df = pd.read_csv(f'{PATH}/CLUSTERED/renos_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(renos_df)

In [6]:
# Load economic data

econ_df = pd.read_csv(f'{PATH}/ECONOMIC/PROCESSED/full_economic_data.csv')

In [7]:
# Load geographic GEOJSON file

gdf = gpd.read_file("C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA/GEOGRAPHIC/PROCESSED/nbhds_with_zones.geojson")

# Assign correct coordinate system
gdf = gdf.set_crs('EPSG:3857', allow_override=True)

# Convert to desired coordinate system
gdf = gdf.to_crs(epsg=4326)

In [8]:
# Get set of permit neighborhoods

builds_nbhds = set(builds_df['nbhd'])

demos_nbhds = set(demos_df['nbhd'])

renos_nbhds = set(renos_df['nbhd'])

permits_nbhds = builds_nbhds.union(demos_nbhds,renos_nbhds)

print(f"There are {len(permits_nbhds)} neighborhoods in our permits df: {permits_nbhds}")

There are 25 neighborhoods in our permits df: {'Downtown Eastside/Strathcona', 'Kerrisdale', 'Hastings/Sunrise/Grandview/Woodlands', 'Sunset', 'Collingwood', 'Kitsilano/Point Grey South', 'Point Grey', 'Renfrew', 'Downtown Central', 'Kitsilano/Point Grey North', 'Downtown North', 'Marpole Remainder', 'Marpole South', 'South Cambie', 'West End/Stanley Park South', 'South False Creek', 'Cedar Cottage', 'English Bay', 'Fraser View/Killarny', 'Riley Park', 'West End/Stanley Park North', 'Mount Pleasant', 'North False Creek', 'Westside/Kerrisdale Remainder', 'South Granville'}


In [9]:
# Restrict gdf nbhds

gdf = gdf[gdf["nbhd"].isin(permits_nbhds)].copy()

gdf_nbhds = set(gdf['nbhd'])

print(f"There are {len(gdf_nbhds)} neighborhoods in geographic boundary df: {gdf_nbhds}")

There are 25 neighborhoods in geographic boundary df: {'Downtown Eastside/Strathcona', 'Kerrisdale', 'Hastings/Sunrise/Grandview/Woodlands', 'Sunset', 'Collingwood', 'Kitsilano/Point Grey South', 'Point Grey', 'Renfrew', 'Downtown Central', 'Kitsilano/Point Grey North', 'Downtown North', 'Marpole South', 'Marpole Remainder', 'South Cambie', 'West End/Stanley Park South', 'South False Creek', 'Cedar Cottage', 'English Bay', 'Fraser View/Killarny', 'Riley Park', 'West End/Stanley Park North', 'Mount Pleasant', 'North False Creek', 'Westside/Kerrisdale Remainder', 'South Granville'}


In [10]:
# Restrict economic nbhds

econ_df["nbhd"] = econ_df["nbhd"].str.title()

econ_df = econ_df[econ_df["nbhd"].isin(permits_nbhds)].copy()

econ_nbhds = set(econ_df['nbhd'])

print(f"There are {len(econ_nbhds)} neighborhoods in our economic df: {econ_nbhds}")

There are 25 neighborhoods in our economic df: {'Downtown Eastside/Strathcona', 'Kerrisdale', 'Hastings/Sunrise/Grandview/Woodlands', 'Collingwood', 'Sunset', 'Kitsilano/Point Grey South', 'Point Grey', 'Renfrew', 'Downtown Central', 'Kitsilano/Point Grey North', 'Downtown North', 'Marpole South', 'Marpole Remainder', 'South Cambie', 'West End/Stanley Park South', 'South False Creek', 'Cedar Cottage', 'English Bay', 'Fraser View/Killarny', 'Riley Park', 'West End/Stanley Park North', 'Mount Pleasant', 'North False Creek', 'Westside/Kerrisdale Remainder', 'South Granville'}


## Geographic Boundary File

In [45]:
# Check gdf

map_boundaries(gdf, 'nbhd', 'geometry','Vancouver Neighborhoods')

In [46]:
# Drop unnecessary 'zone' column in gdf

gdf = gdf.drop(columns = ['zone'])

examine_df(gdf)



Number of records in the dataframe is: 25

The columns in the dataframe are: Index(['nbhd', 'geometry'], dtype='object')


 Other info about dataframe:

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 25 entries, 0 to 24
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   nbhd      25 non-null     object  
 1   geometry  25 non-null     geometry
dtypes: geometry(1), object(1)
memory usage: 600.0+ bytes


None


 Basic statistical info about dataframe:



,nbhd,geometry
count,25,25
unique,25,25
top,West End/Stanley Park North,POLYGON ((-123.1401966853795 49.29038401235166...
freq,1,1




Sample of records in the dataframe:


,nbhd,geometry
0,West End/Stanley Park North,"POLYGON ((-123.1402 49.29038, -123.13862 49.28..."
1,West End/Stanley Park South,"POLYGON ((-123.15862 49.31669, -123.14986 49.3..."
2,English Bay,"POLYGON ((-123.14182 49.28727, -123.14035 49.2..."
3,Downtown Central,"POLYGON ((-123.13 49.2975, -123.12915 49.29642..."
4,North False Creek,"POLYGON ((-123.11718 49.27553, -123.11715 49.2..."


In [51]:
# Save as GeoJSON for Streamlit incorporation

gdf.to_file(
    r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\DATA\nbhds.geojson", 
    driver="GeoJSON"
)

## Economic Data

In [18]:
# Drop the "zone" column and all columns ending with "_change"

econ_clean = econ_df.drop(columns=["zone"] + [c for c in econ_df.columns if c.endswith("_change")])


examine_df(econ_clean)



Number of records in the dataframe is: 200

The columns in the dataframe are: Index(['nbhd', 'year', 'avg_rent_studio', 'avg_rent_one_bedroom',
       'avg_rent_two_bedroom', 'avg_rent_three_bedroom_plus', 'avg_rent_total',
       'med_rent_studio', 'med_rent_one_bedroom', 'med_rent_two_bedroom',
       'med_rent_three_bedroom_plus', 'med_rent_total', 'vacancy_rate_studio',
       'vacancy_rate_one_bedroom', 'vacancy_rate_two_bedroom',
       'vacancy_rate_three_bedroom_plus', 'vacancy_rate_total'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 16 to 527
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   nbhd                             200 non-null    object 
 1   year                             200 non-null    int64  
 2   avg_rent_studio                  200 non-null    float64
 3   avg_rent_one_bedroom   

None


 Basic statistical info about dataframe:



,year,avg_rent_studio,avg_rent_one_bedroom,avg_rent_two_bedroom,avg_rent_three_bedroom_plus,avg_rent_total,med_rent_studio,med_rent_one_bedroom,med_rent_two_bedroom,med_rent_three_bedroom_plus,med_rent_total,vacancy_rate_studio,vacancy_rate_one_bedroom,vacancy_rate_two_bedroom,vacancy_rate_three_bedroom_plus,vacancy_rate_total
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,2020.500000,1433.886757,1699.584667,2349.474922,2990.612123,1831.694376,1428.367586,1669.726689,2290.328812,2802.709359,1744.377909,1.111000,0.996000,1.172500,0.955500,1.078000
std,2.297038,238.496475,275.765823,454.400684,879.776966,298.232623,271.044870,296.983556,456.515309,873.160103,282.702608,1.482623,1.172458,1.348103,1.994407,1.149226
min,2017.000000,860.611765,1090.237184,1273.303749,1620.312268,1099.565417,866.588235,1049.426167,1282.631982,1611.452181,1049.426167,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2018.750000,1280.568814,1533.192985,2011.806691,2227.300337,1651.663374,1246.394052,1479.499733,1942.461763,2050.224215,1584.123185,0.000000,0.400000,0.200000,0.000000,0.500000
50%,2020.500000,1423.379402,1669.517472,2327.864527,2866.107516,1811.852557,1420.061847,1636.876787,2275.949792,2586.000000,1718.796992,0.800000,0.700000,0.800000,0.000000,0.700000
75%,2022.250000,1572.119152,1846.535975,2664.864350,3566.103346,1999.240053,1556.000000,1824.682130,2568.211898,3489.678742,1869.628647,1.500000,1.200000,1.700000,1.300000,1.300000
max,2024.000000,2167.000000,2536.631300,3630.000000,5741.766816,2725.000000,2482.000000,2500.000000,3672.991584,5154.111406,2525.000000,9.200000,9.200000,9.600000,15.100000,9.200000




Sample of records in the dataframe:


,nbhd,year,avg_rent_studio,avg_rent_one_bedroom,avg_rent_two_bedroom,avg_rent_three_bedroom_plus,avg_rent_total,med_rent_studio,med_rent_one_bedroom,med_rent_two_bedroom,med_rent_three_bedroom_plus,med_rent_total,vacancy_rate_studio,vacancy_rate_one_bedroom,vacancy_rate_two_bedroom,vacancy_rate_three_bedroom_plus,vacancy_rate_total
16,Cedar Cottage,2017,1521.609412,1301.675294,1733.176471,2020.047059,1473.797647,1703.294118,1158.240000,1553.882353,1912.470588,1434.352941,0.7,0.5,0.4,0.0,0.5
17,Cedar Cottage,2018,1412.061209,1462.200459,1888.967100,2151.323642,1583.467483,1515.837796,1358.423871,1632.440704,1985.747513,1515.837796,0.5,0.3,0.7,0.0,0.4
18,Cedar Cottage,2019,1477.300448,1353.147982,1770.026906,2470.520179,1503.497758,1495.524664,1252.914798,1708.520179,2050.224215,1480.717489,0.7,0.9,0.4,0.0,0.7
19,Cedar Cottage,2020,1511.536059,1488.874349,1855.994052,2646.887732,1595.384387,1642.973978,1473.011152,1812.936803,2096.208178,1642.973978,1.9,1.3,2.1,0.0,2.0
20,Cedar Cottage,2021,1647.207575,1479.600874,1881.412964,2847.093955,1634.997815,1498.470503,1442.971595,1775.965040,2166.677349,1664.967225,0.0,0.9,0.6,0.0,0.6


In [19]:
# Save as Parquet for Streamlit incorporation

econ_clean.to_parquet(
    r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\DATA\econ.parquet",
    index=False
)

## Permits Data

In [21]:
def parse_point_wkt(s):
    
    """
    Parses point to extract lat and lon
    """
    
    lon, lat = re.findall(r"-?\d+\.\d+", s)
    return float(lon), float(lat)

In [24]:
def normalize_permits(df, permit_type, cluster_prefix):

    """
    Normalize permit data to prepare for Streamline
    """
    
    out = df.copy()
    
    # Ensure datetime
    if "issue_date" in out.columns:
        out["issue_date"] = pd.to_datetime(out["issue_date"], errors="coerce")

    # Lon/Lat from WKT
    lonlat = out["geom"].apply(parse_point_wkt)
    out["lon"] = lonlat.apply(lambda t: t[0])
    out["lat"] = lonlat.apply(lambda t: t[1])

    # Cluster label with prefix; keep original numeric cluster
    out["cluster"] = pd.to_numeric(out["cluster"], errors="coerce").astype("Int64")
    out["cluster_label"] = cluster_prefix + out["cluster"].astype(str)

    # Add permit type
    out["type"] = permit_type

    return out

In [23]:
# Examine raw builds dataframe

examine_df(builds_df)



Number of records in the dataframe is: 9462

The columns in the dataframe are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite',
       'permit_category_new_build_low_density_housing',
       'permit_category_new_build_standalone_laneway', 'cluster'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9462 entries, 0 to 9461
Data columns (total 17 columns):
 #   Column                                         Non-Null Count  Dtype         
---  ------                                         --------------  -----         
 0   issue_date                                     9462 non-null   datetime64[ns]
 1   project_description                            9462 non-null   object        
 2   geom 

None


 Basic statistical info about dataframe:



,issue_date,project_value,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,cluster
count,9462,9.462000e+03,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000
mean,2020-11-29 03:23:19.365884672,1.959642e+06,0.059290,0.392517,0.096703,0.001163,0.004122,0.063940,0.138026,0.225639,0.692560,0.212957,3.106637
min,2017-01-04 00:00:00,1.019777e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2018-08-24 00:00:00,2.387938e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000
50%,2020-09-24 12:00:00,7.068476e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000
75%,2023-01-17 00:00:00,9.782168e+05,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,5.000000
max,2025-05-09 00:00:00,3.498087e+08,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,7.000000
std,NaN,9.530098e+06,0.236179,0.488337,0.295568,0.034078,0.064072,0.244659,0.344945,0.418025,0.461458,0.409419,2.071581




Sample of records in the dataframe:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,cluster
0,2017-04-12,Low Density Housing - New Building - To constr...,POINT (-123.0438851 49.2545202),2.501153e+05,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,0,1,1,0,3
1,2017-09-28,Low Density Housing - New Building - To constr...,POINT (-123.0846679 49.2365573),1.838661e+05,Sunset,Southeast Vancouver,0,1,0,0,0,0,0,0,0,1,3
2,2017-11-27,Low Density Housing - New Building - To constr...,POINT (-123.0342549 49.254328),8.857129e+05,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,1,0,1,0,2
3,2024-12-06,Low Density Housing - New Building - To constr...,POINT (-123.0629329 49.2168193),9.618750e+05,Fraser View/Killarny,Southeast Vancouver,1,0,0,0,0,0,0,0,1,0,5
4,2024-05-03,Low Density Housing - New Building - To constr...,POINT (-123.051063 49.2669312),1.056125e+06,Hastings/Sunrise/Grandview/Woodlands,East Hastings,1,0,0,0,0,0,0,0,1,0,6


In [26]:
# Normalize and examine builds dataframe

norm_builds_df = normalize_permits(builds_df, 'build', 'B')

examine_df(norm_builds_df)



Number of records in the dataframe is: 9462

The columns in the dataframe are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite',
       'permit_category_new_build_low_density_housing',
       'permit_category_new_build_standalone_laneway', 'cluster', 'lon', 'lat',
       'cluster_label', 'type'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9462 entries, 0 to 9461
Data columns (total 21 columns):
 #   Column                                         Non-Null Count  Dtype         
---  ------                                         --------------  -----         
 0   issue_date                                     9462 non-null   datetime64[ns]
 1   project_description                       

None


 Basic statistical info about dataframe:



,issue_date,project_value,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,cluster,lon,lat
count,9462,9.462000e+03,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.000000,9462.0,9462.000000,9462.000000
mean,2020-11-29 03:23:19.365884672,1.959642e+06,0.059290,0.392517,0.096703,0.001163,0.004122,0.063940,0.138026,0.225639,0.692560,0.212957,3.106637,-123.098323,49.242853
min,2017-01-04 00:00:00,1.019777e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-123.223021,49.204501
25%,2018-08-24 00:00:00,2.387938e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.0,-123.139480,49.226523
50%,2020-09-24 12:00:00,7.068476e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.0,-123.086664,49.242303
75%,2023-01-17 00:00:00,9.782168e+05,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,5.0,-123.054213,49.256475
max,2025-05-09 00:00:00,3.498087e+08,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,7.0,-123.023671,49.292483
std,NaN,9.530098e+06,0.236179,0.488337,0.295568,0.034078,0.064072,0.244659,0.344945,0.418025,0.461458,0.409419,2.071581,0.052431,0.019541




Sample of records in the dataframe:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,...,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,permit_category_new_build_low_density_housing,permit_category_new_build_standalone_laneway,cluster,lon,lat,cluster_label,type
0,2017-04-12,Low Density Housing - New Building - To constr...,POINT (-123.0438851 49.2545202),2.501153e+05,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,...,0,0,1,1,0,3,-123.043885,49.254520,B3,build
1,2017-09-28,Low Density Housing - New Building - To constr...,POINT (-123.0846679 49.2365573),1.838661e+05,Sunset,Southeast Vancouver,0,1,0,0,...,0,0,0,0,1,3,-123.084668,49.236557,B3,build
2,2017-11-27,Low Density Housing - New Building - To constr...,POINT (-123.0342549 49.254328),8.857129e+05,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,...,0,1,0,1,0,2,-123.034255,49.254328,B2,build
3,2024-12-06,Low Density Housing - New Building - To constr...,POINT (-123.0629329 49.2168193),9.618750e+05,Fraser View/Killarny,Southeast Vancouver,1,0,0,0,...,0,0,0,1,0,5,-123.062933,49.216819,B5,build
4,2024-05-03,Low Density Housing - New Building - To constr...,POINT (-123.051063 49.2669312),1.056125e+06,Hastings/Sunrise/Grandview/Woodlands,East Hastings,1,0,0,0,...,0,0,0,1,0,6,-123.051063,49.266931,B6,build


In [27]:
# Normalize and examine demos dataframe

norm_demos_df = normalize_permits(demos_df,  permit_type="demo",  cluster_prefix="D")

examine_df(norm_demos_df)



Number of records in the dataframe is: 5714

The columns in the dataframe are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite', 'cluster',
       'lon', 'lat', 'cluster_label', 'type'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5714 entries, 0 to 5713
Data columns (total 19 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   issue_date                         5714 non-null   datetime64[ns]
 1   project_description                5714 non-null   object        
 2   geom                               5714 non-null   object        
 3   project_value                      5714 non-nul

None


 Basic statistical info about dataframe:



,issue_date,project_value,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster,lon,lat
count,5714,5.714000e+03,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.000000,5714.0,5714.000000,5714.000000
mean,2021-02-03 01:38:17.094854656,2.693202e+04,0.019251,0.002275,0.049527,0.002625,0.013651,0.016451,0.700385,0.190060,1.19356,-123.100603,49.244127
min,2017-01-03 00:00:00,1.109978e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-123.223021,49.206164
25%,2018-10-04 00:00:00,1.562543e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-123.140107,49.228796
50%,2021-05-11 00:00:00,1.708520e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.0,-123.089844,49.244068
75%,2023-02-08 18:00:00,1.792941e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.0,-123.056534,49.257708
max,2025-05-09 00:00:00,6.500000e+06,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,-123.023697,49.292228
std,NaN,1.215656e+05,0.137418,0.047648,0.216986,0.051173,0.116046,0.127213,0.458129,0.392382,1.282032,0.051463,0.019173




Sample of records in the dataframe:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster,lon,lat,cluster_label,type
0,2022-11-14,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0677731 49.2249324),15625.427204,Fraser View/Killarny,Southeast Vancouver,0,0,0,0,0,0,1,0,3,-123.067773,49.224932,D3,demo
1,2017-08-23,Field Review - Demolition / Deconstruction - T...,POINT (-123.1211653 49.2588709),75669.289412,South Granville,South Granville/Oak,0,0,0,0,0,1,0,0,0,-123.121165,49.258871,D0,demo
2,2018-08-03,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0894871 49.239411),17490.436113,Sunset,Southeast Vancouver,0,0,0,0,0,0,0,1,3,-123.089487,49.239411,D3,demo
3,2024-12-06,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0629329 49.2168193),15000.000000,Fraser View/Killarny,Southeast Vancouver,0,0,0,0,0,0,1,0,1,-123.062933,49.216819,D1,demo
4,2021-02-11,Low Density Housing - Demolition / Deconstruct...,POINT (-123.0271394 49.2367555),16649.672251,Collingwood,Southeast Vancouver,0,0,0,0,0,0,1,0,3,-123.027139,49.236756,D3,demo


In [28]:
# Normalize and examine renos dataframe

norm_renos_df  = normalize_permits(renos_df,  permit_type="reno",  cluster_prefix="R")

examine_df(norm_renos_df)



Number of records in the dataframe is: 7875

The columns in the dataframe are: Index(['issue_date', 'project_description', 'geom', 'project_value', 'nbhd',
       'zone', 'duplex_w_secondary_suite', 'laneway_house', 'duplex',
       'multiple_conversion_dwelling', 'dwelling_unit', 'multiple_dwelling',
       'single_detached_house', 'single_detached_house_w_sec_suite', 'cluster',
       'lon', 'lat', 'cluster_label', 'type'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7875 entries, 0 to 7874
Data columns (total 19 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   issue_date                         7875 non-null   datetime64[ns]
 1   project_description                7875 non-null   object        
 2   geom                               7875 non-null   object        
 3   project_value                      7875 non-nul

None


 Basic statistical info about dataframe:



,issue_date,project_value,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster,lon,lat
count,7875,7.875000e+03,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.000000,7875.0,7875.000000,7875.000000
mean,2020-12-23 12:28:04.114285568,9.536062e+04,0.001016,0.003810,0.026032,0.018159,0.049270,0.501968,0.252698,0.131556,1.093714,-123.115842,49.260131
min,2017-01-03 00:00:00,5.053050e+02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-123.221795,49.203559
25%,2018-10-30 00:00:00,1.594619e+04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-123.140681,49.246028
50%,2020-12-04 00:00:00,4.000000e+04,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.0,-123.124897,49.264460
75%,2023-01-28 12:00:00,9.434814e+04,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.0,-123.086775,49.277410
max,2025-05-09 00:00:00,6.000000e+06,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5.0,-123.023797,49.293818
std,NaN,2.235591e+05,0.031859,0.061608,0.159240,0.133534,0.216445,0.500028,0.434587,0.338028,1.426679,0.042146,0.021939




Sample of records in the dataframe:


,issue_date,project_description,geom,project_value,nbhd,zone,duplex_w_secondary_suite,laneway_house,duplex,multiple_conversion_dwelling,dwelling_unit,multiple_dwelling,single_detached_house,single_detached_house_w_sec_suite,cluster,lon,lat,cluster_label,type
0,2018-03-06,Field Review - Addition / Alteration - #308\n\...,POINT (-123.156148 49.2705801),26818.668707,Kitsilano/Point Grey North,Kitsilano/Point Grey,0,0,0,0,0,1,0,0,0,-123.156148,49.270580,R0,reno
1,2021-03-11,Field Review - Addition / Alteration - Exterio...,POINT (-123.0571046 49.2611421),221995.630007,Renfrew,Mount Pleasant/Renfrew Heights,0,0,0,0,0,0,1,0,1,-123.057105,49.261142,R1,reno
2,2022-03-01,Field Review - Addition / Alteration - Interio...,POINT (-123.0894788 49.2616599),119794.941900,Mount Pleasant,Mount Pleasant/Renfrew Heights,0,0,0,1,0,0,0,0,0,-123.089479,49.261660,R0,reno
3,2017-03-20,Field Review - Addition / Alteration - Exterio...,POINT (-123.0742693 49.2220461),60961.195294,Sunset,Southeast Vancouver,0,0,0,0,0,0,0,1,3,-123.074269,49.222046,R3,reno
4,2021-10-28,Field Review - Addition / Alteration - Exterio...,POINT (-123.0263275 49.2088643),58828.841952,Fraser View/Killarny,Southeast Vancouver,0,0,0,0,0,1,0,0,0,-123.026327,49.208864,R0,reno


In [34]:
# Save builds, demos, and renos as parquets for Streamlit

norm_builds_df.to_parquet(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\DATA\builds.parquet", index=False)
norm_demos_df.to_parquet(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\DATA\demos.parquet", index=False)
norm_renos_df.to_parquet(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\DATA\renos.parquet", index=False)

In [35]:
# Construct single dataframe with all permit types

keep = ["issue_date","nbhd","lon","lat","type","cluster","cluster_label","project_value"]

permits_all = pd.concat(
    [
        norm_builds_df[keep].copy(),
        norm_demos_df[keep].copy(),
        norm_renos_df[keep].copy()
    ],
    ignore_index=True
)

# Cleanups 
permits_all["issue_date"] = pd.to_datetime(permits_all["issue_date"], errors="coerce")
permits_all = permits_all.dropna(subset=["lon","lat"])  # Ensure mappable points
permits_all = permits_all.sort_values("issue_date")

In [36]:
# Save as a parquet for Streamlit

outpath = r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\DATA\permits.parquet"

permits_all.to_parquet(outpath, index=False) 

## Helper functions

In [12]:
# Define function to examine dataframes

def examine_df(df,
               name = 'dataframe',
               include_stats = True,
               include_sample = True):
    
    """
    Check basic info about a dataframe df
    """
    
    print(f"\n\nNumber of records in the {name} is: {len(df)}\n")
    print(f"The columns in the {name} are: {df.columns}\n")
    print(f"\n Other info about {name}:\n")
    display(df.info())
    if include_stats == True:
        print(f'\n Basic statistical info about {name}:\n')
        display(df.describe())
    if include_sample == True:
        print(f"\n\nSample of records in the {name}:")
        display(df.head(5))

In [13]:
# Define function to map geographic boundaries for some geographic dataframe

def map_boundaries(gdf, name_col, geo_col, title):
    
    """
    Map geographic regions with boundaries
    """
    
    # Prepare geographic dataframe for conversion into json format
    new_gdf = gdf.copy()    
    new_gdf = new_gdf.set_geometry(geo_col)
    new_gdf['dummy'] = [i for i in range(len(new_gdf))]
    new_gdf = new_gdf.to_crs("EPSG:4326")

    # Use geopandas to generate a valid geojson
    geojson = new_gdf.set_index(name_col).__geo_interface__
    geojson = json.loads(new_gdf.to_json())
    
    # Plot
    fig = px.choropleth_map(
                data_frame = new_gdf,
                   geojson = geojson,
                 locations = name_col,
                     color = "dummy",
              featureidkey = f"properties.{name_col}",  # GeoJSON path
                 map_style = "carto-positron",
                    center = {"lat": 49.25, "lon": -123.1},
                      zoom = 8,
                   opacity = 0.6,
                     title = title
    )
    fig.update_traces(marker_line_width=0.5, marker_line_color='black')
    
    fig.update_layout(margin={"r":0,"t":30,"l":0,"b":0})
    
    fig.show()


In [14]:
# Define function to choropleth showing economic metrics by neighborhood changing over time

def animated_econ_choropleth(econ_df: pd.DataFrame, 
                             nbhds_gdf: gpd.GeoDataFrame, 
                             metric: str,
                            color_scale: str = "RdYlGn_r"):
    
    """
    Creates an animated choropleth showing neighborhood polygons colored by a chosen economic metric over years
    """

    # Ensure polygons are in correct coordinate system
    nbhds_gdf = nbhds_gdf.to_crs("EPSG:4326").copy()
    nbhds_gdf["nbhd"] = nbhds_gdf["nbhd"].astype(str).str.strip().str.lower()

    # Prepare tidy econ_df
    econ = econ_df[["nbhd", "year", metric]].copy()
    econ["nbhd"] = econ["nbhd"].astype(str).str.strip().str.lower()
    econ.rename(columns={metric: "value"}, inplace=True)

    # Build a full grid of all years & all neighborhoods
    years = pd.Series(sorted(econ["year"].unique()), name="year")
    nbhds = nbhds_gdf[["nbhd"]].drop_duplicates()
    base = years.to_frame().assign(key=1).merge(nbhds.assign(key=1), on="key").drop(columns="key")

    # Merge the metric values
    plot_df = base.merge(econ, on=["nbhd", "year"], how="left")

    # Fix color range across frames
    vmin = float(plot_df["value"].quantile(0.02))
    vmax = float(plot_df["value"].quantile(0.98))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = 0.0, 1.0  # fallback

    # Build GeoJSON
    nbhds_geojson = json.loads(nbhds_gdf[["nbhd", "geometry"]].to_json())

    # Compute map center
    minx, miny, maxx, maxy = nbhds_gdf.total_bounds
    center = {"lat": (miny + maxy) / 2, "lon": (minx + maxx) / 2}

    # Plot
    fig = px.choropleth_map(
        plot_df.sort_values(["year", "nbhd"]),
        geojson=nbhds_geojson,
        locations="nbhd",
        featureidkey="properties.nbhd",
        color="value",
        animation_frame="year",
        color_continuous_scale = color_scale, 
        range_color=[vmin, vmax],
        height=700,
        title=f"{metric.replace('_', ' ').title()} by Neighborhood Over Time"
    )

    # Layout & styling
    fig.update_layout(
        map_style="carto-positron",
        map_center=center,
        map_zoom=10,
        margin=dict(r=0, t=60, l=0, b=0)
    )
    fig.update_traces(
        marker_line_width=0.5,
        marker_line_color="black",
        hovertemplate="<b>%{location}</b><br>Value: %{z:.2f}<extra></extra>"
    )

    # Slower animation: 1200 ms per frame, 400 ms transition between frames
    fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200
    fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400
    fig.layout.updatemenus[0].buttons[0].args[1]['mode'] = 'immediate'
    
    fig.show()


In [15]:
# Define function to overlaying new building permits and changes in economic metrics

def monthly_permits_over_rent_map(
    econ_df: pd.DataFrame,     
    nbhds_gdf: gpd.GeoDataFrame,
    permits_df: pd.DataFrame, 
    metric: str = "avg_rent_total",
    color_scale: str = "RdYlGn_r"         # green=low, red=high
):
    
    """
    Generate one animated map combining:
      - Choropleth polygons colored by a yearly economic metric 
      - Scatter points for new building permits per month
    """
    
    # Copy to avoid altering originals
    econ = econ_df.copy()
    nb = nbhds_gdf.copy()
    per = permits_df.copy()

    # Ensure datetime
    per["issue_date"] = pd.to_datetime(per["issue_date"])

    # Lowercase neighborhood keys for matching
    for df in [econ,nb,per]:
        df["nbhd"] = df["nbhd"].astype(str).str.strip().str.lower()

    # Ensure polygons in correct coordinate system
    if nb.crs is None:
        raise ValueError("nbhds_gdf has no CRS. Set nb.crs (e.g., EPSG:3857) before calling.")
    nb = nb.to_crs("EPSG:4326")

    # All months present in permits
    months = (
        per["issue_date"].dt.to_period("M")
        .sort_values()
        .unique()
    )
    months_df = pd.DataFrame({
        "frame": [str(m) for m in months],  
        "year":  [m.year for m in months]
    })

    # Cross-join months and neighborhoods so polygons render every month
    base = months_df.assign(key=1).merge(nb[["nbhd"]].assign(key=1), on="key").drop(columns="key")

    # Bring in rent metric by nbhd and year, repeated for all months of that year
    if metric not in econ.columns:
        raise ValueError(f"Metric '{metric}' not found in econ_df.")
        
    econ_small = econ[["nbhd", "year", metric]].rename(columns={metric: "value"})
    poly_df = base.merge(econ_small, on=["nbhd", "year"], how="left")

    # Prepare polygons GeoJSON 
    nb_geojson = json.loads(nb[["nbhd", "geometry"]].to_json())

    # Global color range
    vmin = float(poly_df["value"].quantile(0.02))
    vmax = float(poly_df["value"].quantile(0.98))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = 0.0, 1.0

    # Map center
    minx, miny, maxx, maxy = nb.total_bounds
    map_center = {"lat": (miny + maxy)/2, "lon": (minx + maxx)/2}

    # Build polygons figure (animated by month)
    fig_poly = px.choropleth_map(
        poly_df.sort_values(["frame", "nbhd"]),
        geojson=nb_geojson,
        locations="nbhd",
        featureidkey="properties.nbhd",
        color="value",
        animation_frame="frame",
        color_continuous_scale=color_scale,  
        range_color=[vmin, vmax],
        height=750,
        title=f"{metric.replace('_',' ').title()} by Neighborhood (shaded) + New Building Permits (points)"
    )
    fig_poly.update_layout(
        map_style="carto-positron",
        map_center=map_center,
        map_zoom=10,
        margin=dict(r=0, t=60, l=0, b=0),
        coloraxis_colorbar=dict(title=metric.replace('_',' ').title())
    )
    fig_poly.update_traces(
        marker_line_width=0.6,
        marker_line_color="black",
        hovertemplate="<b>%{location}</b><br>Value: %{z:.2f}<extra></extra>"
    )

    # Keep only new building permits
    per = per.loc[per["type_of_work_new_building"] == 1].copy()

    # Extract lon/lat from geom column
    coords = per["geom"].str.extract(
        r'POINT\s*\(\s*(?P<lon>-?\d+\.?\d*)\s+(?P<lat>-?\d+\.?\d*)\s*\)'
    ).astype(float)
    per["lon"] = coords["lon"]
    per["lat"] = coords["lat"]
    per = per.dropna(subset=["lat", "lon"])

    # Monthly frame key
    per["frame"] = per["issue_date"].dt.to_period("M").astype(str)

    # Size by project value (with soft clipping to keep dots reasonable)
    q99 = per["project_value"].quantile(0.99) if "project_value" in per.columns else None
    if q99 and np.isfinite(q99):
        per["project_value_vis"] = per["project_value"].clip(upper=float(q99))
    else:
        per["project_value_vis"] = per.get("project_value", 1.0)

    # Build permit scatter plot figure
    fig_pts = px.scatter_map(
        per.sort_values("frame"),
        lat="lat",
        lon="lon",
        size="project_value_vis",
        size_max=22,
        animation_frame="frame",
        hover_name=per["nbhd"] if "nbhd" in per.columns else None,
        hover_data={
            "project_value": ":.0f" if "project_value" in per.columns else False,
            "issue_date": True,
            "lat": False,
            "lon": False,
            "project_value": False
        },
    )
    fig_pts.update_traces(marker=dict(opacity=0.72))  # subtle

    # Add the initial scatter traces to base figure
    fig = fig_poly
    for tr in fig_pts.data:
        fig.add_trace(tr)

    # Merge frames by name so both layers animate together
    frames_poly = {fr.name: fr for fr in fig.frames}
    for fr in fig_pts.frames:
        if fr.name in frames_poly:
            frames_poly[fr.name].data += fr.data
        else:
            # If a month exists in points but not in polygons, create it
            fig.frames += (fr,)

    # Control animation pace
    # 1100 ms per frame hold, 400 ms transition
    if fig.layout.updatemenus and fig.layout.updatemenus[0].buttons:
        fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1100
        fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400
        fig.layout.updatemenus[0].buttons[0].args[1]['mode'] = 'immediate'
    if fig.layout.sliders:
        fig.layout.sliders[0].transition.duration = 400

    return fig


In [16]:
# Define function to map feature prefix to color to enhance visualization interpretability

# Color mapping by prefix
prefix_colors = {
        "avg_rent": "tab:blue",
        "med_rent": "tab:green",
        "vacancy_rate": "tab:red",}

def color_for(col):

    """
    Define color mapping functin
    """
    
    for p, c in prefix_colors.items():
        if col.startswith(p):
            return c
    if col.endswith("_change"):
        return "gray"
    return "black"

In [17]:
# Define function to plot average economic metrics over time

def plot_economic_metrics_grid(df, nbhd = None):

    """
    Plot a grid of subplots for economic metrics over time, averaged across neighborhoods
    """

    # Filter by neighborhood if requested
    if nbhd is not None:
        subset = df[df["nbhd"] == nbhd].copy()
        if subset.empty:
            raise ValueError(f"No rows found for neighborhood: {nbhd!r}")
        title_prefix = f"Neighborhood: {nbhd.title()}"
    else:
        subset = df.copy()
        title_prefix = "All neighborhoods (mean)"

    # Select metric columns: numeric, excluding identifiers and *_change
    exclude_cols = ["nbhd", "zone", "year"]
    metrics = [
        c for c in subset.columns
        if c not in exclude_cols
        and np.issubdtype(subset[c].dtype, np.number)
        and not c.endswith("_change")
    ]
    if not metrics:
        raise ValueError("No numeric metric columns found to plot.")

    # Aggregate by year
    yearly = subset.groupby("year", as_index=False)[metrics].mean().sort_values("year")

    # Grid setup
    n = len(metrics)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(ncols * 4.5, nrows * 3),
        sharex=False
    )
    axes = np.array(axes).reshape(-1)
    
    years = yearly["year"].values

    # Plot
    for i, metric in enumerate(metrics):
        ax = axes[i]
        ax.plot(years, yearly[metric].values, marker="o", linewidth=1.5, color=color_for(metric))
        ax.set_title(metric.replace("_", " ").title(), fontsize=9)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.set_xlabel("Year")
        ax.set_xticks(years)
        ax.tick_params(axis="x", rotation=45)

    # Hide unused axes
    for j in range(len(metrics), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Economic Metrics Over Time — {title_prefix}", fontsize=12, y=0.995)
    fig.tight_layout(rect=[0, 0.0, 1, 0.97])

    return fig, axes